In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle as pkl

from pdpbox import pdp
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv", index_col=0)
y_train = pd.read_csv("../data/processed/y_train.csv", index_col=0)

X_test = pd.read_csv("../data/processed/X_test.csv", index_col=0)
y_test = pd.read_csv("../data/processed/y_test.csv", index_col=0)

In [ ]:
def clean_data(X):
    """
    Drops columns with no predictive value:
    - Patient_ID: only an identifier
    - Diagnosis_Year/Date: redundant temporal info
    - Survival_Months: target leakage. 
    """
    X = X.drop(['Patient_ID', 'Diagnosis_Year', 
                              'Diagnosis_Date', 'Survival_Months'], axis=1)
    return X

In [ ]:
# removing columns again to reduce correlation instead of using PCA
def engineer_X(X):

    # adding pack_years as a feature (a standard clinical metric)
    X['Pack_Years'] = (X['Cigarettes_Per_Day'] / 20) * X['Years_Smoking']

    # changing cancer stage to ordinal values
    stage_map = {'Stage I': 1,
                 'Stage II': 2,
                 'Stage III': 3,
                 'Stage IV': 4}
    X['Cancer_Stage_Numeric'] = X['Cancer_Stage'].map(stage_map)

    # changing binary values to 1 and 0
    binary_cols = ['Secondhand_Smoke', 'Family_History', 'Occupational_Hazard',
                'Chronic_Lung_Disease', 'Asbestos_Exposure', 'Radon_Exposure',
                'Previous_Cancer_History', 'Coughing','Shortness_of_Breath', 
                'Chest_Pain', 'Coughing_Blood', 'Fatigue', 'Weight_Loss', 
                'Wheezing', 'Recurrent_Infections', 'Swallowing_Difficulty', 
                'Finger_Clubbing', 'Metastasis']

    X[binary_cols] = X[binary_cols].eq("Yes").astype(int)

    # converting gender values
    X['Gender'] = X['Gender'].eq("Male").astype(int)
    
    # converting cancer types
    X['Cancer_Type'] = X['Cancer_Type'].eq("NSCLC").astype(int)

    # creating a count column for symptoms and risk
    symptom_cols = ['Coughing','Shortness_of_Breath', 'Chest_Pain', 
                    'Coughing_Blood', 'Fatigue', 'Weight_Loss', 'Wheezing', 
                    'Recurrent_Infections', 'Swallowing_Difficulty', 
                    'Finger_Clubbing', 'Metastasis']
    X['Symptom_Count'] = X[symptom_cols].sum(axis=1).astype(int)

    risk_cols = ['Secondhand_Smoke', 'Family_History', 'Occupational_Hazard',
                'Chronic_Lung_Disease', 'Asbestos_Exposure', 'Radon_Exposure',
                'Previous_Cancer_History']
    X['Risk_Count'] = X[risk_cols].sum(axis=1).astype(int)

    X = X.drop(columns=['Cigarettes_Per_Day', 'Years_Smoking', 
                        'Cancer_Stage', 'Tumor_Size_cm', 'Country',
                        'WHO_Region'])

    return X


In [ ]:
def bool_target(y):
    """Converts target column from 'Yes'/'No' to 1/0."""
    y['Survived'] = y['Survived'].eq("Yes").astype(int)
    return y

In [ ]:
# Apply cleaning, feature engineering, and target encoding to both splits.
X_train = engineer_X(clean_data(X_train))
X_test = engineer_X(clean_data(X_test))

y_train = bool_target(y_train)
y_test = bool_target(y_test)

In [ ]:
# defining numeric, categorical and binary columns
num_cols = ['Age', 'BMI', 'Pack_Years', 'Symptom_Count', 'Risk_Count']

cat_cols = ['Smoking_Status', 'Air_Pollution_Exposure', 'Alcohol_Use', 
            'Exercise_Frequency', 'Genetic_Mutation', 'NSCLC_Subtype', 
             'Diagnosis_Method', 'Treatment']

binary_cols = ['Gender', 'Secondhand_Smoke', 'Family_History', 
               'Occupational_Hazard', 'Chronic_Lung_Disease', 
               'Asbestos_Exposure', 'Radon_Exposure', 
               'Previous_Cancer_History', 'Coughing','Shortness_of_Breath', 
               'Chest_Pain', 'Coughing_Blood', 'Fatigue', 'Weight_Loss', 
               'Wheezing', 'Recurrent_Infections', 'Swallowing_Difficulty', 
               'Finger_Clubbing', 'Metastasis', 'Cancer_Type']


# Random Forest with Hyperparameter tuning

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('ord', OrdinalEncoder(categories=[[1, 2, 3, 4]]), ['Cancer_Stage_Numeric']), 
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('bin', 'passthrough', binary_cols)
])

rf = RandomForestClassifier(class_weight='balanced', # to compensate for resampling
                            random_state=0)

param_grid = {
      'clf__n_estimators': [50, 100, 150],                               
      'clf__max_depth': [None, 10, 20],                                   
      'clf__min_samples_split': [2, 5, 7],
      'clf__min_samples_leaf': [8, 10, 12, 16],
  }

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', rf)
])

grid_search = GridSearchCV(estimator=rf_pipeline,
                           param_grid=param_grid,
                           scoring='f1',
                           cv=5,
                           n_jobs=-1,
                           verbose=1)


In [ ]:
grid_search.fit(X_train, y_train.values.ravel())

print("Best params:", grid_search.best_params_)
print(f"Best CV F1:  {grid_search.best_score_:.4f}")

In [ ]:
y_pred = grid_search.predict(X_test)
y_prob = grid_search.predict_proba(X_test)[:, 1]

print(f"Random Forest with Hyperparameter tuning Performance:\n")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, 
                                      y_prob):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

In [ ]:
y_pred = grid_search.predict(X_test)
y_prob = grid_search.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, 
                      y_pred, 
                      labels=[0, 1])

disp = ConfusionMatrixDisplay(cm, 
                              display_labels=['No', 'Yes']).plot(cmap='Blues')


In [ ]:
# prepare a list to append results
results = []
results.append({
    'Model':'Random Forest',
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred)
})

# XGBoost

In [ ]:
# to reuse preprocessor from random forest
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('ord', OrdinalEncoder(categories=[[1, 2, 3, 4]]), ['Cancer_Stage_Numeric']), 
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('bin', 'passthrough', binary_cols)
])

xgb = XGBClassifier(random_state=0,
                    eval_metric='logloss',
                    verbosity=0)

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', xgb)])

param_grid_xgb = {
      'clf__n_estimators': [200, 250, 300],
      'clf__max_depth': [1, 3, 5],
      'clf__learning_rate': [0.001, 0.01, 0.1],
      'clf__subsample': [0.8, 1.0],
      'clf__colsample_bytree': [0.8, 1.0],
  }

xgb_grid_search = GridSearchCV(estimator=xgb_pipeline,
                               param_grid=param_grid_xgb,
                               scoring='f1',
                               cv=5,
                               n_jobs=-1,
                               verbose=1)

In [ ]:
xgb_grid_search.fit(X_train, y_train.values.ravel())

print("Best params:", xgb_grid_search.best_params_)     
print(f"Best CV F1: {xgb_grid_search.best_score_:.4f}")

In [ ]:
y_pred = xgb_grid_search.predict(X_test)
y_prob = xgb_grid_search.predict_proba(X_test)[:, 1]

print(f"XGBoost Performance:\n")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, 
                                      y_prob):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

In [ ]:
# prepare a list to append results
results.append({
    'Model':'XGBoost',
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred)
})

# Logistic Regression 

In [ ]:
from math import e


lr = LogisticRegression(random_state=0,
                        class_weight='balanced',
                        max_iter=1000)

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', lr)
])

param_grid_lr = {                                             
      'clf__C': [0.01, 0.1, 1, 10],                  
      'clf__penalty': ['l1', 'l2'],                       
      'clf__solver': ['liblinear', 'saga'],
  }  

lr_grid_search = GridSearchCV(lr_pipeline,
                              param_grid_lr,
                              scoring='f1',
                              cv=5,
                              n_jobs=-1,
                              verbose=1)



In [ ]:
lr_grid_search.fit(X_train, y_train.values.ravel())

print("Best params.:", lr_grid_search.best_params_)
print(f"Best CV F1: {lr_grid_search.best_score_:.4f}")

In [ ]:
y_pred = lr_grid_search.predict(X_test)
y_prob = lr_grid_search.predict_proba(X_test)[:, 1]

print(f"Logistic Regression Performance:\n")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, 
                                      y_prob):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")

In [ ]:
# prepare a list to append results
results.append({
    'Model':'Logistic Regression',
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred)
})

In [ ]:
# displaying results as a dataframe
df_results = pd.DataFrame(results).set_index('Model').round(4)
df_results

In [ ]:
best_rf = grid_search.best_estimator_

X_names = (
    num_cols + 
    ['Cancer_Stage_Numeric'] + 
    list(best_rf.named_steps['preprocessor'].named_transformers_['cat']
         .get_feature_names_out(cat_cols))
         + binary_cols)

importances = best_rf.named_steps['clf'].feature_importances_
feat_imp = pd.Series(importances, index=X_names).sort_values(ascending=False)

feat_imp.head(15).plot(kind='barh', figsize=(8, 6))
plt.gca().invert_yaxis()
plt.title('Top 15 Important Features - Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
with open('../data/processed/best_rf_model.pkl', 'wb') as f:
    pkl.dump(grid_search.best_estimator_, f)

In [ ]:
model_features = X_train.columns.tolist()       

for feature, title in [('BMI', 'BMI'),          
('Cancer_Stage_Numeric', 'Cancer Stage')]:
      pdp_feat = pdp.PDPIsolate(                  
          model=best_rf,
          df=X_train,
          model_features=model_features,
          feature=feature,
          feature_name=title,
          n_classes=2,
      )
      fig, axes = pdp_feat.plot()
      plt.suptitle(f'PDP - {title}')
      plt.tight_layout()
      plt.show()
      display(fig)